# Simple: Merge Datasets with PAMOLA.CORE

**Goal**: Learn dataset merging basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply dataset merging using execute()
- Compare before/after results
- Understand join types and relationships

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 01_merge_datasets_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd 
import numpy as np 
import sys 
import os 
import json 
from pathlib import Path 
from datetime import datetime 
from IPython.display import Image, display 
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.merging.merge_datasets_op import MergeDatasetsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `customers.csv` and `orders.csv`
- Auto-creates sample data if files missing
- Review both dataset structures before proceeding

**What you'll see:**
- Customers dataset: 10 records with customer_id, name, email, city
- Orders dataset: 20 records with order_id, customer_id, product, amount
- Key field analysis showing matching and non-matching records
- Relationship: one-to-many (one customer can have multiple orders)

**Note:** Sample demonstrates left join scenario - 2 customers without orders (IDs 9-10)

In [ ]:
examples_dir = project_root / 'examples'
customers_path = examples_dir / 'data_examples' / 'customers.csv'
orders_path = examples_dir / 'data_examples' / 'orders.csv'

# Create customers data if it doesn't exist
if not customers_path.exists():
    print("⚠️  Customers file not found, creating sample data...")
    customers_path.parent.mkdir(parents=True, exist_ok=True)
    
    customers_data = pd.DataFrame({
        'customer_id': range(1, 11),
        'customer_name': [
            'Alice Johnson', 'Bob Smith', 'Carol White', 'David Brown', 'Eve Davis',
            'Frank Miller', 'Grace Wilson', 'Henry Moore', 'Iris Taylor', 'Jack Anderson'
        ],
        'email': [
            'alice@email.com', 'bob@email.com', 'carol@email.com', 'david@email.com', 'eve@email.com',
            'frank@email.com', 'grace@email.com', 'henry@email.com', 'iris@email.com', 'jack@email.com'
        ],
        'city': [
            'New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix',
            'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose'
        ],
        'join_date': pd.date_range('2023-01-01', periods=10, freq='ME')
    })
    customers_data.to_csv(customers_path, index=False)
    print(f"✓ Customers data created")

# Create orders data if it doesn't exist
if not orders_path.exists():
    print("⚠️  Orders file not found, creating sample data...")
    orders_path.parent.mkdir(parents=True, exist_ok=True)
    
    np.random.seed(42)
    orders_data = pd.DataFrame({
        'order_id': range(1, 21),
        'customer_id': np.random.choice(range(1, 9), 20),  # Only customers 1-8 have orders
        'order_date': pd.date_range('2024-01-01', periods =20, freq='W'),
        'product': np.random.choice(['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard'], 20),
        'amount': np.random.randint(50, 2000, 20),
        'status': np.random.choice(['Completed', 'Pending', 'Shipped'], 20)
    })
    orders_data.to_csv(orders_path, index=False)
    print(f"✓ Orders data created")

# Load both datasets
customers_df = read_csv(customers_path)
orders_df = read_csv(orders_path)

print(f"\n📊 CUSTOMERS Dataset: {len(customers_df)} records, {len(customers_df.columns)} columns")
print(f"\n🔍 First 5 customers:")
print(customers_df.head())

print(f"\n📊 ORDERS Dataset: {len(orders_df)} records, {len(orders_df.columns)} columns")
print(f"\n🔍 First 5 orders:")
print(orders_df.head())

print(f"\n📈 Dataset Statistics:")
print(f"\n  CUSTOMERS:")
for col in customers_df.columns:
    dtype_str = str(customers_df[col].dtype)
    if pd.api.types.is_string_dtype(customers_df[col]):
        print(f"    {col:20s} ({dtype_str:10s}): {customers_df[col].nunique()} unique values")
    else:
        print(f"    {col:20s} ({dtype_str:10s})")

print(f"\n  ORDERS:")
for col in orders_df.columns:
    dtype_str = str(orders_df[col].dtype)
    if pd.api.types.is_string_dtype(orders_df[col]):
        print(f"    {col:20s} ({dtype_str:10s}): {orders_df[col].nunique()} unique values")
    else:
        print(f"    {col:20s} ({dtype_str:10s})")

# Show key field analysis
if 'customer_id' in customers_df.columns:
    print(f"\n🔑 Key Field Analysis (customer_id):")
    print(f"  Customers with customer_id: {customers_df['customer_id'].nunique()} unique")
    print(f"  Orders with customer_id: {orders_df['customer_id'].nunique()} unique")
    print(f"  Customers with orders: {len(set(customers_df['customer_id']) & set(orders_df['customer_id']))}")
    print(f"  Customers without orders: {len(set(customers_df['customer_id']) - set(orders_df['customer_id']))}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE merge parameters** in `create_config_kwargs()`
   - `left_dataset_name`: Main dataset (e.g., "customers")
   - `right_dataset_name`: Dataset to join (e.g., "orders")
   - `left_key` & `right_key`: Common field(s) for joining
2. Run to validate configuration and setup environment

**What you'll see:**
- Left dataset with key field and unique count
- Right dataset with key field and unique count
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource created with 2 datasets (✓)

**Note:** Keys must exist in both datasets - can have different names (left_key ≠ right_key)

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "left_dataset_name": "customers",
        "right_dataset_name": "orders",
        "left_key": "customer_id",
        "right_key": "customer_id"  # Can be different from left_key
    }

kwargs = create_config_kwargs()
left_key = kwargs["left_key"]
right_key = kwargs["right_key"]

# Validate keys exist
print(f"\n🔍 Validating merge configuration...\n")
if left_key not in customers_df.columns:
    raise ValueError(
        f"❌ Left key '{left_key}' not found in customers dataset!\n"
        f"Available columns: {', '.join(customers_df.columns)}\n"
        f"Please update 'left_key' in create_config_kwargs()"
    )

if right_key not in orders_df.columns:
    raise ValueError(
        f"❌ Right key '{right_key}' not found in orders dataset!\n"
        f"Available columns: {', '.join(orders_df.columns)}\n"
        f"Please update 'right_key' in create_config_kwargs()"
    )

print(f"✓ Left dataset: '{kwargs['left_dataset_name']}'")
print(f"  Key field: '{left_key}'")
print(f"  Records: {len(customers_df)}")
print(f"  Unique keys: {customers_df[left_key].nunique()}")

print(f"\n✓ Right dataset: '{kwargs['right_dataset_name']}'")
print(f"  Key field: '{right_key}'")
print(f"  Records: {len(orders_df)}")
print(f"  Unique keys: {orders_df[right_key].nunique()}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="transformation",
    description="Merge customers and orders datasets",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {
    "dataset_name": kwargs["left_dataset_name"]
}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource with both datasets
data_source = DataSource(
    dataframes={
        "customers": customers_df,
        "orders": orders_df
    }
)
print("✓ DataSource created with 2 datasets")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Merging on {left_key}",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute dataset merge
- Monitor progress bar (6 steps: load → validate → merge → save → complete)

**Key parameters:**
- `join_type='left'`: Keep all left records (inner/right/outer also available)
- `relationship_type='auto'`: Auto-detect one-to-one or one-to-many
- `suffixes=('', '')`: Handle duplicate column names
- Join types: left (all left), inner (matching only), outer (all records)

**What you'll see:**
- Configuration summary with datasets, keys, and join type
- Progress bar: load datasets → validate keys → perform merge → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (3-5 files expected)

**Note:** Left join preserves all left records - right records without matches get NaN values

In [ ]:
# Create operation with production-style configuration
operation = MergeDatasetsOperation(
    # Core parameters
    left_dataset_name=kwargs["left_dataset_name"],
    right_dataset_name=kwargs["right_dataset_name"],
    left_key=left_key,
    right_key=right_key,
    
    # Merge strategy
    join_type='left',           # Keep all customers
    relationship_type='auto',    # Auto-detect relationship
    suffixes=('', ''),      # For duplicate columns
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Left dataset:     {operation.left_dataset_name}")
print(f"  Right dataset:    {operation.right_dataset_name}")
print(f"  Join keys:        {operation.left_key} = {operation.right_key}")
print(f"  Join type:        {operation.join_type}")
print(f"  Relationship:     {operation.relationship_type}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing merge datasets...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load merged data from task_dir
- Review merge statistics and record counts
- Verify join behavior (all customers preserved in left join)

**What you'll see:**
- Sample merged records (first 10 rows)
- Dataset size comparison: customers, orders, merged result
- Merge analysis: customers with/without orders
- Summary: record counts showing join effectiveness
- Key metrics: execution time, match percentage

**Note:** Left join keeps all customers - those without orders have NaN in order fields

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 MERGE RESULTS")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Merged Records (first 10):")
    print(result_df.head(10))
    
    # Dataset size comparison
    print(f"\n📈 Dataset Size Comparison:")
    print(f"{'Dataset':<20} {'Records':>10} {'Fields':>10}")
    print("-" * 45)
    print(f"{'Customers (left)':<20} {len(customers_df):>10} {len(customers_df.columns):>10}")
    print(f"{'Orders (right)':<20} {len(orders_df):>10} {len(orders_df.columns):>10}")
    print(f"{'Merged (result)':<20} {len(result_df):>10} {len(result_df.columns):>10}")
    
    # Analyze merge results
    print(f"\n📊 Merge Analysis:")
    # Count customers with and without orders
    if 'order_id' in result_df.columns:
        customers_with_orders = result_df['order_id'].notna().sum()
        customers_without_orders = result_df['order_id'].isna().sum()
        print(f"  Customers with orders:    {customers_with_orders}")
        print(f"  Customers without orders: {customers_without_orders}")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Left join successfully combined {len(customers_df)} customers")
    print(f"  with {len(orders_df)} orders into {len(result_df)} records")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Left join keeps all customers, even those without orders!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-2 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Merged CSV (combined records)
├── metrics/          # Merge statistics JSON
├── visualizations/   # Match analysis charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Merge statistics and performance metrics
2. **Output Data**: Merged records with sample rows
3. **Visualizations**: Charts showing merge analysis

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # METADATA
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # DATASET METRICS
            print("\n📊 DATASET METRICS:")

            dataset_metrics = [
                ("total_input_records", "Input Records"),
                ("total_output_records", "Output Records"),
                ("total_input_fields", "Input Fields"),
                ("total_output_fields", "Output Fields"),
            ]

            for key, label in dataset_metrics:
                if key in metrics:
                    print(f"   {label:<22}: {metrics[key]}")

            # MERGE STATISTICS
            print("\n🔄 MERGE STATISTICS:")

            merge_metrics = [
                ("num_matched_records", "Matched Records"),
                ("num_only_in_left", "Only in Left"),
                ("num_only_in_right", "Only in Right"),
                ("num_duplicate_keys_left", "Duplicate Keys (Left)"),
                ("num_duplicate_keys_right", "Duplicate Keys (Right)"),
            ]

            for key, label in merge_metrics:
                if key in metrics:
                    print(f"   {label:<28}: {metrics[key]}")

            # Match percentage (special handling)
            if "match_percentage" in metrics:
                print(
                    f"   {'Match Percentage':<28}: "
                    f"{metrics['match_percentage']:.2f}% of input records"
                )

            # FIELD COUNTS (NO GUESSING)
            print("\n📊 FIELD COUNTS:")

            if "num_fields_before" in metrics:
                print(f"   Fields Before Merge   : {metrics['num_fields_before']}")

            if "num_fields_after" in metrics:
                print(f"   Fields After Merge    : {metrics['num_fields_after']}")

            # PERFORMANCE
            print("\n⚡ PERFORMANCE:")

            if "execution_time_seconds" in metrics:
                print(
                    f"   Execution Time        : "
                    f"{metrics['execution_time_seconds']:.4f}s"
                )

            if "records_per_second" in metrics:
                print(
                    f"   Records / Second      : "
                    f"{metrics['records_per_second']:.2f}"
                )
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show column types
            print(f"\n\n📊 Column Overview:")
            print("-" * 80)
            print(f"{'Column':<30} {'Type':<15} {'Non-Null':>10}")
            print("-" * 60)
            for col in output_df.columns:
                dtype = str(output_df[col].dtype)
                non_null = output_df[col].notna().sum()
                print(f"{col:<30} {dtype:<15} {non_null:>10}")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load multiple datasets from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure merge operation with keys and join type  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Left join keeps all left records (customers)
- Auto-detect identifies one-to-many relationships
- Merge combines datasets based on common keys
- Visualizations show overlap and match statistics
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_merge_datasets_advanced.ipynb`** includes:
- All join types (inner, left, right, outer)
- Multiple key fields (composite keys)
- Different relationship types validation
- Handling duplicate keys
- Custom suffix handling
- External file loading
- Performance optimization for large datasets

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🔄 [Merge Operations Guide](../../docs/merge_operations.md)